In [7]:
import pandas as pd
import numpy as np

In [8]:
np.random.seed(1)

In [14]:
df = pd.DataFrame({
    "Subject": np.repeat(["S1", "S2", "S3"], 6),
    "Condition": np.tile(np.repeat(["A", "B"], 3), 3),
    "Session": np.tile([1, 2, 3], 6),
    "RT": np.random.normal(loc=500, scale=50, size=18)
})

> groupby() splits the dataset into subsets according to categorical keys.

In [15]:
# Note: After grouping, aggregation functions summarize each subset.

df.groupby("Condition")["RT"].mean()

Condition
A    488.017004
B    501.924391
Name: RT, dtype: float64

> Aggregation Functions

Common built-in functions:
   - mean()
   - std()
   - count()
   - median()
   - min(), max()

In [21]:
# Multiple aggregations:
print(
    "Multi aggregations:\n",
    df.groupby("Condition")["RT"].agg(["mean", "std", "count"]),
    "\n"
)

# Custom aggregations:
print(
    "Custom aggregations:\n",
    df.groupby("Condition")["RT"].agg(lambda x: x.max() - x.min()),
    "\n"
)

# Name custom outputs:
print(
    "Name custop outputs:\n",
    df.groupby("Condition")["RT"].agg(
    mean_RT="mean",
    sd_RT="std",
    n_trials="count"
)
)

Multi aggregations:
                  mean        std  count
Condition                              
A          488.017004  32.777792      9
B          501.924391  38.552872      9 

Custom aggregations:
 Condition
A    100.073756
B    104.024657
Name: RT, dtype: float64 

Name custop outputs:
               mean_RT      sd_RT  n_trials
Condition                                 
A          488.017004  32.777792         9
B          501.924391  38.552872         9


##### Multi-Index Groups

In [22]:
# Grouping by multiple variables produces a hierarchical (multi-level) index:
print(df.head())
print(df.groupby(["Condition", "Session"])["RT"].mean())

  Subject Condition  Session          RT
0      S1         A        1  502.110687
1      S1         A        2  529.140761
2      S1         A        3  444.969041
3      S1         B        1  557.236185
4      S1         B        2  545.079536
Condition  Session
A          1          504.190149
           2          491.705564
           3          468.155298
B          1          489.395811
           2          499.374275
           3          517.003087
Name: RT, dtype: float64


In [27]:
# Let's compute mean RT per condition.

mean_rt_cond = (
    df.groupby("Condition")["RT"]
    .mean()
)

print(mean_rt_cond) # This reports average RT per experimental condition, collapsing across subjects and sessions.
# Note: this is appropriate for descriptive reporting but does not account for within-subject dependencies.

Condition
A    488.017004
B    501.924391
Name: RT, dtype: float64


In [28]:
# Let's compute SD per subject:

std_rt_sub = (
    df.groupby("Subject")["RT"]
    .std()
)

print(std_rt_sub) # This quantifies within-subject variability.
# Note: it is useful for identifying unstable performers or excessive noise.

Subject
S1    40.076205
S2    35.120839
S3    15.048941
Name: RT, dtype: float64


Note: SD is sensitive to small sample sizes. Few trials = unstable SD.

In [35]:
# Let's create a multi-index (mean, SD, N per cell) summary table (Condition × Session).

summary_table = (
    df.groupby(["Condition","Session"])["RT"]
    .agg(
        Mean_RT="mean",
        SD_RT="std",
        N="count"
    )
)

print(
    "Research-style:\n", # hierarchical index
    summary_table,
    "\n"
)

print(
    "Publication-style:\n",
    summary_table.reset_index(),
    "\n"
)

print(
    "Pivot-style:\n",
    df.groupby(["Condition", "Session"])["RT"]
      .mean()
      .unstack("Session")
)

Research-style:
                       Mean_RT      SD_RT  N
Condition Session                          
A         1        504.190149  39.853626  3
          2        491.705564  33.204170  3
          3        468.155298  24.539990  3
B         1        489.395811  58.795097  3
          2        499.374275  40.846227  3
          3        517.003087  15.289372  3 

Publication-style:
   Condition  Session     Mean_RT      SD_RT  N
0         A        1  504.190149  39.853626  3
1         A        2  491.705564  33.204170  3
2         A        3  468.155298  24.539990  3
3         B        1  489.395811  58.795097  3
4         B        2  499.374275  40.846227  3
5         B        3  517.003087  15.289372  3 

Pivot-style:
 Session             1           2           3
Condition                                    
A          504.190149  491.705564  468.155298
B          489.395811  499.374275  517.003087
